# Generation timeseries

This notebook is used to analze the ramping and the minimum power generation of conventional power plants to better approximate the power generation in the pypsa-eur model.

In [201]:
import pandas as pd
import numpy as np

In [221]:
generation_per_type=pd.read_csv("../data/Generation_per_type_Germany_2019.csv")

In [222]:
generation_per_type.index=pd.to_datetime(generation_per_type["MTU"].apply(lambda x : x.split("-")[0]))

In [223]:
generation_per_type.drop(["Area", "MTU"], axis=1, inplace=True)

In [224]:
generation_per_type.columns=[item for sublist in generation_per_type.columns.str.split("  - ").str[:1] for item in sublist]

### Minimum generation power during 2019 of different power generators

In [206]:
generation_per_type.min()

Biomass                            3880.0
Fossil Brown coal/Lignite          3744.0
Fossil Gas                         1546.0
Fossil Hard coal                    731.0
Fossil Oil                           62.0
Geothermal                            6.0
Hydro Pumped Storage                  0.0
Hydro Pumped Storage                  0.0
Hydro Run-of-river and poundage    1050.0
Hydro Water Reservoir                 1.0
Nuclear                            3371.0
Other                               100.0
Other renewable                     104.0
Solar                                 0.0
Waste                                32.0
Wind Offshore                         0.0
Wind Onshore                        248.0
dtype: float64

### Relative generation power during 2019 of different power generators

In [207]:
generation_per_type.min()/generation_per_type.max()

Biomass                            0.768469
Fossil Brown coal/Lignite          0.213212
Fossil Gas                         0.110153
Fossil Hard coal                   0.043556
Fossil Oil                         0.058879
Geothermal                         0.193548
Hydro Pumped Storage               0.000000
Hydro Pumped Storage               0.000000
Hydro Run-of-river and poundage    0.482094
Hydro Water Reservoir              0.001799
Nuclear                            0.353799
Other                              0.210970
Other renewable                    0.525253
Solar                              0.000000
Waste                              0.035794
Wind Offshore                      0.000000
Wind Onshore                       0.006138
dtype: float64

- relative generation capacity:
    - brown coal = 0.21
    - hard coal = 0.044
    - nuclear = 0.353

### Maximal ramping within 15 minutes
#### For coal in MW

In [208]:
np.abs(generation_per_type["Fossil Brown coal/Lignite"].diff()).max()

1340.0

#### For nuclear in MW

In [209]:
np.abs(generation_per_type["Nuclear"].diff()).max()

1041.0

### Further Analysis with installed capacity and generation per unit

In [216]:
generation_per_unit=pd.read_csv("../data/Generation_per_unit_Germany_2019.csv", header=[0,1], index_col=0)
installed_capacity=pd.read_csv("../data/Capacity_per_type_Germany_2019.csv", index_col=3)
installed_capacity.drop(columns=installed_capacity.columns[0], inplace=True)

In [217]:
generation_per_unit.head()

,GKM AG DBEnergie,GKM AG TNG,HKW Altbach/Deizisau 2,HKW Heilbronn Block 7,RDK 7,RDK 8,KKW Neckarwestheim 2,KKW Philippsburg 2,BERGKAMEN_A,BEXBACH_A_GESAMT,...,KW Jänschwalde Block B,KW Jänschwalde Block C,KW Jänschwalde Block D,KW Jänschwalde Block E,KW Lippendorf Block R,KW Lippendorf Block S,KW Schwarze Pumpe Block A,KW Schwarze Pumpe Block B,Schkopau A,Schkopau B
,Fossil Hard coal,Fossil Hard coal,Fossil Hard coal,Fossil Hard coal,Fossil Hard coal,Fossil Hard coal,Nuclear,Nuclear,Fossil Hard coal,Fossil Hard coal,...,Fossil Brown coal/Lignite,Fossil Brown coal/Lignite,Fossil Brown coal/Lignite,Fossil Brown coal/Lignite,Fossil Brown coal/Lignite,Fossil Brown coal/Lignite,Fossil Brown coal/Lignite,Fossil Brown coal/Lignite,Fossil Brown coal/Lignite,Fossil Brown coal/Lignite
2019-01-01 00:00:00+01:00,30.0,127.0,76.0,389.0,337.0,549.0,1308.0,1363.0,0.0,0.0,...,404.0,0.0,0.0,377.0,489.0,0.0,506.0,387.0,64.0,233.0
2019-01-01 01:00:00+01:00,19.0,79.0,0.0,403.0,208.0,284.0,1311.0,1367.0,0.0,0.0,...,392.0,0.0,0.0,424.0,477.0,0.0,462.0,21.0,0.0,218.0
2019-01-01 02:00:00+01:00,16.0,65.0,0.0,401.0,272.0,108.0,1303.0,1361.0,0.0,0.0,...,435.0,0.0,0.0,455.0,469.0,0.0,492.0,0.0,0.0,219.0
2019-01-01 03:00:00+01:00,13.0,57.0,0.0,660.0,259.0,0.0,987.0,1366.0,0.0,0.0,...,407.0,0.0,0.0,451.0,467.0,0.0,455.0,0.0,0.0,223.0
2019-01-01 04:00:00+01:00,12.0,49.0,0.0,685.0,170.0,0.0,861.0,1364.0,0.0,0.0,...,386.0,0.0,0.0,425.0,462.0,0.0,446.0,0.0,0.0,222.0


In [218]:
installed_capacity.head()

,Bidding Zone,Installed Capacity [MW],Production Type,Voltage Connection Level [kV]
Name,,,,
Vorarlberger Illwerke AG,10YDE-ENBW-----N,2142.0,Hydro Pumped Storage,220
HKW Heilbronn,10YDE-ENBW-----N,778.0,Fossil Hard coal,380
Schluchseewerk AG,10YDE-ENBW-----N,258.0,Hydro Pumped Storage,220
KW Rheinfelden DE,10YDE-ENBW-----N,100.0,Hydro Run-of-river and poundage,220
Rheinkraftwerk Iffezheim,10YDE-ENBW-----N,146.0,Hydro Run-of-river and poundage,110


#### Relative generation where maximum capacity is taken from the maximal installed capacity instead of maximal power generation

In [219]:
max_capacity=installed_capacity.groupby("Production Type").sum()["Installed Capacity [MW]"]

In [225]:
generation_per_type.min().divide(max_capacity)

Biomass                                 NaN
Fossil Brown coal/Lignite          0.197990
Fossil Coal-derived gas                 NaN
Fossil Gas                         0.082935
Fossil Hard coal                   0.033304
Fossil Oil                         0.040181
Geothermal                              NaN
Hydro Pumped Storage               0.000000
Hydro Pumped Storage               0.000000
Hydro Run-of-river and poundage    2.982955
Hydro Water Reservoir              0.002000
Nuclear                            0.353874
Other                                   NaN
Other renewable                         NaN
Solar                                   NaN
Waste                              0.086957
Wind Offshore                      0.000000
Wind Onshore                       0.940463
dtype: float64

- relative generation capacity:
    - brown coal =0.20
    - hard coal =0.033
    - nuclear =0.354

## Script to retrieve the data files for Generation_per_unit_Germany_2019.csv and Capacity_per_type_Germany_2019.csv
Takes a bit longer to generate. Files are already downloaded in the repository

In [ ]:
from entsoe import EntsoePandasClient
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
client = EntsoePandasClient(api_key="94076b59-96cb-481b-8231-0685c805328e", retry_count=20, retry_delay=30)
start = pd.Timestamp('20190101', tz='Europe/Berlin')
end = pd.Timestamp('20200101', tz='Europe/Berlin')
country_codes = ["DE_TRANSNET", 'DE_AMPRION', "DE_TENNET", "DE_50HZ"]
psr_types=["B05", "B14", "B02"] # nuclear, hard coal and lignite
generation_dict=dict()
installed_capacity=dict()
for country_code in country_codes:
    installed_capacity[country_code]=client.query_installed_generation_capacity_per_unit(country_code, start=start,end=end, psr_type=None)
    for psr_type in psr_types:
        try:
            generation_dict[country_code+"_"+psr_type]=client.query_generation_per_plant(country_code, start=start, end=end, psr_type=psr_type)
            print(f"Country code {country_code} and psr type {psr_type} finsihed.")
        except:
            continue
pd.concat([df for df in generation_dict.values()], axis=1).to_csv("../data/Generation_per_unit_Germany_2019.csv")
pd.concat([df for df in installed_capacity.values()], axis=0).to_csv("../data/Capacity_per_type_Germany_2019.csv")